In [1]:
from scipy.stats import norm
from scipy.stats import chi2

from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from statsmodels.genmod.families import Binomial
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.graphics.mosaicplot import mosaic

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from itertools import product
pd.set_option("display.width", 2000)

# import custom functions
from ST5213_helpers import *

In [2]:
df = pd.DataFrame(list(product(['A', 'N'], ['F', 'M'], ['F0', 'F1', 'F1+'], ['avg', 'slow'])), columns=['E', 'S', 'A', 'L'])
df['Count'] = [4, 1, 5, 10, 10, 8, 5, 3, 2, 3, 14, 4,
               4, 1, 6, 11, 11, 9, 6, 3, 2, 7, 14, 3]
df[['E', 'S', 'A', 'L']] = df[['E', 'S', 'A', 'L']].astype('category')
print(df.head())

   E  S    A     L  Count
0  A  F   F0   avg      4
1  A  F   F0  slow      1
2  A  F   F1   avg      5
3  A  F   F1  slow     10
4  A  F  F1+   avg     10


In [3]:
df.pivot(index=['E', 'S', 'A'], columns='L', values='Count').sort_index()

L        avg  slow
E S A             
A F F0     4     1
    F1     5    10
    F1+   10     8
  M F0     5     3
    F1     2     3
    F1+   14     4
N F F0     4     1
    F1     6    11
    F1+   11     9
  M F0     6     3
    F1     2     7
    F1+   14     3

In [4]:
fm = glm('Count ~ E*S*A*L', data=df, family=Poisson()).fit()

print(drop1_glm(fm))

        Df      Deviance         AIC       LRT  Pr(>Chi)
<none>     -2.220781e-15  130.861131                    
E:S:A:L  2  5.117429e-01  127.372874  0.511743  0.774241


c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\regression\_tools.py:133: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
c:\Users\YuxinLi\anaconda3\envs\ST5213\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1269: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  return self._fit_irls(


In [5]:
fm = update_glm(fm, drop='E:S:A:L')

print(drop1_glm(fm))

       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.511743  127.372874                    
E:S:A   2  1.018763  123.879894   0.50702  0.776072
E:S:L   1  0.523314  125.384445  0.011571  0.914339
E:A:L   2  0.646069  123.507200  0.134326  0.935043
S:A:L   2  5.324484  128.185615  4.812741  0.090142


In [6]:
fm = update_glm(fm, drop='E:A:L')

print(drop1_glm(fm))

       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.646069  123.507200                    
E:S:A   2  1.173572  120.034704  0.527504  0.768164
E:S:L   1  0.646881  121.508012  0.000812  0.977266
S:A:L   2  5.502019  124.363150   4.85595  0.088215


In [7]:
fm = update_glm(fm, drop='E:S:L')

print(drop1_glm(fm))

       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.646881  121.508012                    
E:L     1  0.648442  119.509574  0.001562  0.968479
E:S:A   2  1.250584  118.111715  0.603703  0.739448
S:A:L   2  5.521278  122.382410  4.874398  0.087405


In [8]:
fm = update_glm(fm, drop='E:L')

print(drop1_glm(fm))

       Df  Deviance         AIC       LRT  Pr(>Chi)
<none>     0.648442  119.509574                    
E:S:A   2  1.256334  116.117465  0.607892  0.737901
S:A:L   2  5.527104  120.388236  4.878662  0.087219


In [9]:
fm = update_glm(fm, drop='S:A:L')

print(drop1_glm(fm))

       Df   Deviance         AIC        LRT  Pr(>Chi)
<none>      5.527104  120.388236                     
S:L     1   6.709266  119.570397   1.182162  0.276916
A:L     2  19.587451  130.448582  14.060347  0.000885
E:S:A   2   6.134996  116.996127   0.607892  0.737901


In [10]:
fm = update_glm(fm, drop='S:L')

print(drop1_glm(fm))

       Df   Deviance         AIC        LRT  Pr(>Chi)
<none>      6.709266  119.570397                     
A:L     2  22.996037  131.857169  16.286771  0.000291
E:S:A   2   7.317158  116.178289   0.607892  0.737901


In [11]:
print(drop1_glm(fm))

       Df   Deviance         AIC        LRT  Pr(>Chi)
<none>      6.709266  119.570397                     
A:L     2  22.996037  131.857169  16.286771  0.000291
E:S:A   2   7.317158  116.178289   0.607892  0.737901


In [12]:
print(chi2.sf(fm.deviance, fm.df_resid))

0.6673594742789205


In [13]:
print(fm.deviance, fm.df_resid)

6.709266104023781 9


In [14]:
print(fm.summary2())

                  Results: Generalized linear model
Model:                 GLM                 Method:            IRLS    
Model Family:          Poisson             AIC:               119.5704
Link Function:         Log                 BIC:               137.2412
Dependent Variable:    Count               Log-Likelihood:    -44.785 
Date:                  2026-10-27 10:00    LL-Null:           -70.738 
No. Observations:      24                  Deviance:          6.7093  
Df Model:              14                  Pearson chi2:      6.50    
Df Residuals:          9                   Scale:             1.0000  
----------------------------------------------------------------------
                        Coef.  Std.Err.    z    P>|z|   [0.025  0.975]
----------------------------------------------------------------------
Intercept               1.2580   0.4643  2.7094 0.0067  0.3480  2.1681
E[T.N]                  0.0000   0.6325  0.0000 1.0000 -1.2396  1.2396
S[T.M]                  0

In [15]:
df['fitted'] = fm.fittedvalues 
collapsed_fitted = df.pivot_table(index=['A'], columns='L', values='fitted', aggfunc='sum').sort_index()
print(collapsed_fitted)

L     avg  slow
A              
F0   19.0   8.0
F1   15.0  31.0
F1+  49.0  24.0


In [16]:
np.exp(1.5909), 19*31/(15*8)

(np.float64(4.908164288976372), 4.908333333333333)

In [18]:
np.exp(0.1512), 19*24/(49*8)

(np.float64(1.1632292806749205), 1.163265306122449)

In [19]:
df['res']  = fm.resid_pearson / np.sqrt(1-hat_diag(fm))
tab_pearson = df.pivot_table(index=['E', 'S', 'A'], columns='L', values='res').sort_index()
print(tab_pearson.round(3))

L          avg   slow
E S A                
A F F0   0.522 -0.522
    F1   0.073 -0.073
    F1+ -1.204  1.204
  M F0  -0.581  0.581
    F1   0.373 -0.373
    F1+  1.109 -1.109
N F F0   0.522 -0.522
    F1   0.297 -0.297
    F1+ -1.354  1.354
  M F0  -0.298  0.298
    F1  -0.741  0.741
    F1+  1.526 -1.526
